# Milestone 1: Exploratory Data Analysis (EDA)
## EuroSAT RGB: Land-Use and Land-Cover (LULC) Classification

### Project Objective
Analyze the **EuroSAT RGB** dataset containing Sentinel-2 satellite optical imagery tiles across 10 distinct land-use and land-cover classes. This notebook systematically examines:
1. Dataset integrity, directory hierarchy, and class distributions.
2. Channel and pixel intensity statistics (mean, std, min, max).
3. Class-by-class visual inspections.
4. Spectral and spatial similarities among confusable class pairs.
5. Zero-leakage stratified train/val/test split verification.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

from src.constants import (
    EUROSAT_CLASSES,
    CLASS_TO_IDX,
    IDX_TO_CLASS,
    CLASS_COLORS,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    IMAGE_CHANNELS,
)
from src.data.validator import DatasetValidator
from src.data.split import load_splits

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120
print("Environment initialized successfully.")

--- 
## 1. Dataset Integrity & Class Distribution

EuroSAT contains 27,000 geo-referenced image patches covering 13 European countries, collected by the ESA Sentinel-2 satellite.

In [ ]:
DATASET_DIR = RAW_DATA_DIR / "EuroSAT_RGB"
validator = DatasetValidator()
report = validator.validate_dataset(DATASET_DIR)

print(report.validation_summary)

# Create class counts DataFrame
df_counts = pd.DataFrame([
    {"Class": cls_name, "Count": report.class_counts.get(cls_name, 0), "Percentage": (report.class_counts.get(cls_name, 0) / max(1, report.total_images_scanned)) * 100}
    for cls_name in EUROSAT_CLASSES
])
df_counts = df_counts.sort_values(by="Count", ascending=False).reset_index(drop=True)
display(df_counts)

In [ ]:
plt.figure(figsize=(10, 5))
colors = [CLASS_COLORS.get(c, "#4C72B0") for c in df_counts["Class"]]
bars = plt.barh(df_counts["Class"], df_counts["Count"], color=colors, edgecolor="black", linewidth=0.8)

plt.xlabel("Number of Images", fontsize=11, fontweight="bold")
plt.ylabel("Land-Use / Land-Cover Class", fontsize=11, fontweight="bold")
plt.title("EuroSAT RGB Class Distribution (Total = 27,000 Tiles)", fontsize=13, fontweight="bold", pad=12)
plt.gca().invert_yaxis()

for bar in bars:
    w = bar.get_width()
    plt.text(w + 30, bar.get_y() + bar.get_height()/2, f"{int(w):,}", va="center", fontsize=9, fontweight="bold")

plt.xlim(0, max(df_counts["Count"]) + 400)
plt.tight_layout()
plt.show()

--- 
## 2. Visual Inspection: Sample Tiles Across All 10 Classes

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
axes = axes.flatten()

for idx, cls_name in enumerate(EUROSAT_CLASSES):
    cls_dir = DATASET_DIR / cls_name
    images = list(cls_dir.glob("*.jpg"))
    if images:
        sample_img = Image.open(images[0])
        axes[idx].imshow(sample_img)
    axes[idx].set_title(f"{cls_name}\n({len(images):,} tiles)", fontsize=10, fontweight="bold", color=CLASS_COLORS.get(cls_name, "black"))
    axes[idx].axis("off")

plt.suptitle("Sample Satellite Image Tiles for Each EuroSAT Class (64x64 RGB)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

--- 
## 3. Pixel Statistics & Channel Distribution

Computing dataset-wide mean and standard deviation per channel for Sentinel-2 optical data:

In [ ]:
# Sample a representative subset across all classes for statistical profiling
pixel_samples = []
sample_limit = 100  # 100 images per class = 1,000 images sample

for cls_name in EUROSAT_CLASSES:
    cls_dir = DATASET_DIR / cls_name
    for img_p in list(cls_dir.glob("*.jpg"))[:sample_limit]:
        arr = np.array(Image.open(img_p)) / 255.0  # normalize to [0, 1]
        pixel_samples.append(arr)

pixel_array = np.array(pixel_samples)  # Shape: (N, 64, 64, 3)
r_channel = pixel_array[:, :, :, 0].flatten()
g_channel = pixel_array[:, :, :, 1].flatten()
b_channel = pixel_array[:, :, :, 2].flatten()

print(f"Empirical Channel Statistics (Sampled across {len(pixel_samples)} tiles):")
print(f"  Red Channel   - Mean: {r_channel.mean():.4f}, Std: {r_channel.std():.4f}")
print(f"  Green Channel - Mean: {g_channel.mean():.4f}, Std: {g_channel.std():.4f}")
print(f"  Blue Channel  - Mean: {b_channel.mean():.4f}, Std: {b_channel.std():.4f}")

# Plot histograms
plt.figure(figsize=(10, 4))
plt.hist(r_channel, bins=60, color="red", alpha=0.5, density=True, label="Red Channel")
plt.hist(g_channel, bins=60, color="green", alpha=0.5, density=True, label="Green Channel")
plt.hist(b_channel, bins=60, color="blue", alpha=0.5, density=True, label="Blue Channel")
plt.title("RGB Channel Pixel Intensity Distribution (EuroSAT Normalized [0, 1])", fontsize=12, fontweight="bold")
plt.xlabel("Normalized Pixel Intensity", fontsize=10)
plt.ylabel("Density", fontsize=10)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

--- 
## 4. Confusable Class Pairs & Visual Challenges

Satellite imagery exhibits unique challenges, notably high intra-class variance and subtle inter-class boundaries:

In [ ]:
confusing_pairs = [
    ("AnnualCrop", "PermanentCrop", "Agricultural texture vs. orchard row structures"),
    ("HerbaceousVegetation", "Pasture", "Natural grassland vs. managed grazing land"),
    ("Residential", "Industrial", "High building density with small vs. large roof footprints"),
    ("Highway", "River", "Linear dark/grey transport corridors vs. meandering waterways"),
]

fig, axes = plt.subplots(len(confusing_pairs), 4, figsize=(12, 3.2 * len(confusing_pairs)))

for row_idx, (cls_a, cls_b, note) in enumerate(confusing_pairs):
    imgs_a = list((DATASET_DIR / cls_a).glob("*.jpg"))[:2]
    imgs_b = list((DATASET_DIR / cls_b).glob("*.jpg"))[:2]
    
    axes[row_idx, 0].imshow(Image.open(imgs_a[0]))
    axes[row_idx, 0].set_title(f"{cls_a} (Sample 1)", fontsize=9, fontweight="bold")
    axes[row_idx, 0].axis("off")
    
    axes[row_idx, 1].imshow(Image.open(imgs_a[1]))
    axes[row_idx, 1].set_title(f"{cls_a} (Sample 2)", fontsize=9, fontweight="bold")
    axes[row_idx, 1].axis("off")
    
    axes[row_idx, 2].imshow(Image.open(imgs_b[0]))
    axes[row_idx, 2].set_title(f"{cls_b} (Sample 1)", fontsize=9, fontweight="bold")
    axes[row_idx, 2].axis("off")
    
    axes[row_idx, 3].imshow(Image.open(imgs_b[1]))
    axes[row_idx, 3].set_title(f"{cls_b} (Sample 2)\n[{note}]", fontsize=8, color="#C0392B")
    axes[row_idx, 3].axis("off")

plt.suptitle("Comparative Analysis of Confusable Land-Cover Classes", fontsize=13, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

--- 
## 5. Stratified Split Manifest Verification

Verifying that the train (70%), validation (15%), and test (15%) splits have strictly **zero data leakage** and identical class proportions.

In [ ]:
splits_file = PROCESSED_DATA_DIR / "splits.json"
if splits_file.exists():
    splits_data = load_splits(splits_file)
    print(f"Methodology: {splits_data['methodology']}")
    print(f"Total Samples : {splits_data['total_samples']:,}")
    print(f"Train Samples : {splits_data['train_count']:,} ({splits_data['train_count']/splits_data['total_samples']*100:.1f}%)")
    print(f"Val Samples   : {splits_data['val_count']:,} ({splits_data['val_count']/splits_data['total_samples']*100:.1f}%)")
    print(f"Test Samples  : {splits_data['test_count']:,} ({splits_data['test_count']/splits_data['total_samples']*100:.1f}%)")
    
    df_split = pd.DataFrame(splits_data["class_distribution"]).T
    display(df_split)
else:
    print("Splits file not found. Run 'python scripts/create_splits.py' to generate.")

--- 
## 6. Key Takeaways for Modeling & Engineering

1. **Rotational Invariance**: Aerial satellite imagery is naturally rotation-invariant. Flips and $90^\circ/180^\circ/270^\circ$ rotations during training will substantially improve generalization.
2. **Spatial Resolution ($64 \times 64$)**: Receptive fields in CNNs need to capture both macro patterns (e.g. geometric roofs for Residential/Industrial) and micro texture (e.g. canopy grain in Forests, rows in Crops).
3. **Metric Selection**: Macro F1 is essential. Imbalanced classes like `Pasture` (2,000 samples) vs `AnnualCrop` (3,000 samples) must not distort overall model performance evaluation.
4. **Grad-CAM Role**: Will be critical for debugging whether the network attends to actual roads in `Highway` or background vegetation.